# 04 Offline Augmentation

Generate realistic offline IR, harsh sunlight, and blur/compression variants from training images only.


## Purpose

This notebook creates additional training samples after Notebook 03 has split the original dataset. It reads only from `data/splits_original/train`, copies YOLO labels unchanged, and writes augmented outputs under `data/augmented_train`.

Validation and test folders are never read for augmentation and remain original.


## Setup


In [1]:
from pathlib import Path
import sys

import matplotlib.pyplot as plt
import pandas as pd
from PIL import Image
import yaml
from IPython.display import display


def find_v2_root(start: Path) -> Path:
    """Find the v2_pipeline root from the current working directory."""
    for candidate in (start, *start.parents):
        if candidate.name == "v2_pipeline" and (candidate / "src").exists():
            return candidate
        nested = candidate / "ppe-detection" / "v2_pipeline"
        if nested.exists() and (nested / "src").exists():
            return nested
    raise RuntimeError("Could not find v2_pipeline root")


V2_ROOT = find_v2_root(Path.cwd().resolve())
if str(V2_ROOT) not in sys.path:
    sys.path.insert(0, str(V2_ROOT))

from src.augmentation.offline_blur_compression import (
    generate_blur_compression_augmentation,
)
from src.augmentation.offline_ir import generate_ir_augmentation
from src.augmentation.offline_sunlight import generate_sunlight_augmentation

V2_ROOT

WindowsPath('C:/Github/smart-factory-safety-monitoring-system/ppe-detection/v2_pipeline')

This setup cell makes the notebook portable. It finds the `v2_pipeline` folder from whatever working directory Jupyter starts in, adds that folder to `sys.path`, and imports only the reusable augmentation functions from `src/augmentation`.

Keeping the image-processing logic in `src/` makes the notebook easier to review and lets the same functions be smoke-tested outside Jupyter.


## Load Configuration


In [2]:
dataset_config_path = V2_ROOT / "configs" / "dataset_config.yaml"
augmentation_config_path = V2_ROOT / "configs" / "augmentation_config.yaml"

with dataset_config_path.open("r", encoding="utf-8") as file_handle:
    dataset_config = yaml.safe_load(file_handle)
with augmentation_config_path.open("r", encoding="utf-8") as file_handle:
    augmentation_config = yaml.safe_load(file_handle)

splits_original_dir = V2_ROOT / dataset_config["splits_original_dir"]
augmented_train_dir = V2_ROOT / dataset_config["augmented_train_dir"]
train_images_dir = splits_original_dir / "train" / "images"
train_labels_dir = splits_original_dir / "train" / "labels"
random_seed = int(dataset_config.get("random_seed", 42))

offline_config = augmentation_config["offline_augmentation"]
ir_ratio = float(offline_config["ir_ratio"])
sunlight_ratio = float(offline_config["sunlight_ratio"])
blur_compression_ratio = float(offline_config["blur_compression_ratio"])

config_summary = pd.DataFrame(
    [
        {"setting": "ir_ratio", "value": ir_ratio},
        {"setting": "sunlight_ratio", "value": sunlight_ratio},
        {"setting": "blur_compression_ratio", "value": blur_compression_ratio},
        {"setting": "random_seed", "value": random_seed},
    ]
)
display(config_summary)

,setting,value
0,ir_ratio,0.25
1,sunlight_ratio,0.15
2,blur_compression_ratio,0.10
3,random_seed,42.00


This cell centralizes all paths and ratios from YAML files instead of hardcoding local machine paths. The ratios control how many original training images are sampled for each offline augmentation type.

The same `random_seed` used by the dataset pipeline is reused here so the selected augmentation subset is deterministic and reproducible.


## Check Training Split


In [3]:
valid_image_extensions = {".jpg", ".jpeg", ".png"}

if not train_images_dir.exists() or not train_labels_dir.exists():
    raise RuntimeError("Training split is missing. Run Notebook 03 before Notebook 04.")

train_image_paths = sorted(
    path
    for path in train_images_dir.iterdir()
    if path.is_file() and path.suffix.lower() in valid_image_extensions
)
train_label_paths = sorted(
    path
    for path in train_labels_dir.iterdir()
    if path.is_file() and path.suffix.lower() == ".txt"
)

if not train_image_paths:
    raise RuntimeError(
        "Training split is empty. Run Notebook 03 and confirm train images exist."
    )

missing_labels = [
    image_path.name
    for image_path in train_image_paths
    if not (train_labels_dir / f"{image_path.stem}.txt").exists()
]
if missing_labels:
    print(
        "Warning: some selected training images may be skipped due to missing labels."
    )
    print(missing_labels[:10])

print(f"Original train images: {len(train_image_paths)}")
print(f"Original train labels: {len(train_label_paths)}")

Original train images: 380
Original train labels: 380


This is the safety gate for the notebook. It confirms Notebook 03 has produced a non-empty train split before any augmentation is attempted.

Only `splits_original/train` is inspected here. Validation and test folders are deliberately ignored so evaluation data stays original and untouched.


## Intended Augmentation Counts


In [4]:
def expected_count(num_images: int, ratio: float) -> int:
    if ratio == 0:
        return 0
    return min(num_images, max(1, int(round(num_images * ratio))))


intended_counts = pd.DataFrame(
    [
        {
            "augmentation": "ir",
            "ratio": ir_ratio,
            "expected_images": expected_count(len(train_image_paths), ir_ratio),
        },
        {
            "augmentation": "sunlight",
            "ratio": sunlight_ratio,
            "expected_images": expected_count(len(train_image_paths), sunlight_ratio),
        },
        {
            "augmentation": "blur_compression",
            "ratio": blur_compression_ratio,
            "expected_images": expected_count(
                len(train_image_paths), blur_compression_ratio
            ),
        },
    ]
)
display(intended_counts)

,augmentation,ratio,expected_images
0,ir,0.25,95
1,sunlight,0.15,57
2,blur_compression,0.10,38


This preview estimates how many images each augmentation will try to create before any files are written. It uses the configured ratios and the number of original train images.

For non-zero ratios, very small datasets still produce at least one selected sample, while larger datasets follow the configured percentage.


## Generate Augmentations

By default, existing augmented files are skipped. Set `overwrite_existing_augmented = True` only when intentionally regenerating outputs.


In [5]:
overwrite_existing_augmented = False

ir_report = generate_ir_augmentation(
    images_dir=train_images_dir,
    labels_dir=train_labels_dir,
    output_images_dir=augmented_train_dir / "ir" / "images",
    output_labels_dir=augmented_train_dir / "ir" / "labels",
    ratio=ir_ratio,
    seed=random_seed,
    overwrite=overwrite_existing_augmented,
)

sunlight_report = generate_sunlight_augmentation(
    images_dir=train_images_dir,
    labels_dir=train_labels_dir,
    output_images_dir=augmented_train_dir / "sunlight" / "images",
    output_labels_dir=augmented_train_dir / "sunlight" / "labels",
    ratio=sunlight_ratio,
    seed=random_seed,
    overwrite=overwrite_existing_augmented,
)

blur_compression_report = generate_blur_compression_augmentation(
    images_dir=train_images_dir,
    labels_dir=train_labels_dir,
    output_images_dir=augmented_train_dir / "blur_compression" / "images",
    output_labels_dir=augmented_train_dir / "blur_compression" / "labels",
    ratio=blur_compression_ratio,
    seed=random_seed,
    overwrite=overwrite_existing_augmented,
)

display(ir_report.head())
display(sunlight_report.head())
display(blur_compression_report.head())

,augmentation_type,original_image_path,original_label_path,augmented_image_path,augmented_label_path,status,notes
0,ir,C:\Github\smart-factory-safety-monitoring-syst...,C:\Github\smart-factory-safety-monitoring-syst...,C:\Github\smart-factory-safety-monitoring-syst...,C:\Github\smart-factory-safety-monitoring-syst...,generated,
1,ir,C:\Github\smart-factory-safety-monitoring-syst...,C:\Github\smart-factory-safety-monitoring-syst...,C:\Github\smart-factory-safety-monitoring-syst...,C:\Github\smart-factory-safety-monitoring-syst...,generated,
2,ir,C:\Github\smart-factory-safety-monitoring-syst...,C:\Github\smart-factory-safety-monitoring-syst...,C:\Github\smart-factory-safety-monitoring-syst...,C:\Github\smart-factory-safety-monitoring-syst...,generated,
3,ir,C:\Github\smart-factory-safety-monitoring-syst...,C:\Github\smart-factory-safety-monitoring-syst...,C:\Github\smart-factory-safety-monitoring-syst...,C:\Github\smart-factory-safety-monitoring-syst...,generated,
4,ir,C:\Github\smart-factory-safety-monitoring-syst...,C:\Github\smart-factory-safety-monitoring-syst...,C:\Github\smart-factory-safety-monitoring-syst...,C:\Github\smart-factory-safety-monitoring-syst...,generated,


,augmentation_type,original_image_path,original_label_path,augmented_image_path,augmented_label_path,status,notes
0,sunlight,C:\Github\smart-factory-safety-monitoring-syst...,C:\Github\smart-factory-safety-monitoring-syst...,C:\Github\smart-factory-safety-monitoring-syst...,C:\Github\smart-factory-safety-monitoring-syst...,generated,
1,sunlight,C:\Github\smart-factory-safety-monitoring-syst...,C:\Github\smart-factory-safety-monitoring-syst...,C:\Github\smart-factory-safety-monitoring-syst...,C:\Github\smart-factory-safety-monitoring-syst...,generated,
2,sunlight,C:\Github\smart-factory-safety-monitoring-syst...,C:\Github\smart-factory-safety-monitoring-syst...,C:\Github\smart-factory-safety-monitoring-syst...,C:\Github\smart-factory-safety-monitoring-syst...,generated,
3,sunlight,C:\Github\smart-factory-safety-monitoring-syst...,C:\Github\smart-factory-safety-monitoring-syst...,C:\Github\smart-factory-safety-monitoring-syst...,C:\Github\smart-factory-safety-monitoring-syst...,generated,
4,sunlight,C:\Github\smart-factory-safety-monitoring-syst...,C:\Github\smart-factory-safety-monitoring-syst...,C:\Github\smart-factory-safety-monitoring-syst...,C:\Github\smart-factory-safety-monitoring-syst...,generated,


,augmentation_type,original_image_path,original_label_path,augmented_image_path,augmented_label_path,status,notes
0,blur_compression,C:\Github\smart-factory-safety-monitoring-syst...,C:\Github\smart-factory-safety-monitoring-syst...,C:\Github\smart-factory-safety-monitoring-syst...,C:\Github\smart-factory-safety-monitoring-syst...,generated,
1,blur_compression,C:\Github\smart-factory-safety-monitoring-syst...,C:\Github\smart-factory-safety-monitoring-syst...,C:\Github\smart-factory-safety-monitoring-syst...,C:\Github\smart-factory-safety-monitoring-syst...,generated,
2,blur_compression,C:\Github\smart-factory-safety-monitoring-syst...,C:\Github\smart-factory-safety-monitoring-syst...,C:\Github\smart-factory-safety-monitoring-syst...,C:\Github\smart-factory-safety-monitoring-syst...,generated,
3,blur_compression,C:\Github\smart-factory-safety-monitoring-syst...,C:\Github\smart-factory-safety-monitoring-syst...,C:\Github\smart-factory-safety-monitoring-syst...,C:\Github\smart-factory-safety-monitoring-syst...,generated,
4,blur_compression,C:\Github\smart-factory-safety-monitoring-syst...,C:\Github\smart-factory-safety-monitoring-syst...,C:\Github\smart-factory-safety-monitoring-syst...,C:\Github\smart-factory-safety-monitoring-syst...,generated,


This cell performs the actual offline augmentation. Each function samples from the train images, applies a mild visual-only transform, saves the augmented image, and copies the matching YOLO label without changing its contents.

The `overwrite_existing_augmented` flag is intentionally `False` by default. If augmented files already exist, the functions skip those outputs and record the skip in the report instead of silently replacing work.


## Save Reports


In [6]:
reports_dir = V2_ROOT / "reports" / "augmentation"
reports_dir.mkdir(parents=True, exist_ok=True)

ir_report_path = reports_dir / "ir_augmentation_report.csv"
sunlight_report_path = reports_dir / "sunlight_augmentation_report.csv"
blur_report_path = reports_dir / "blur_compression_augmentation_report.csv"

ir_report.to_csv(ir_report_path, index=False)
sunlight_report.to_csv(sunlight_report_path, index=False)
blur_compression_report.to_csv(blur_report_path, index=False)

all_reports = pd.concat(
    [ir_report, sunlight_report, blur_compression_report],
    ignore_index=True,
)
summary_df = (
    all_reports.groupby(["augmentation_type", "status"], dropna=False)
    .size()
    .reset_index(name="count")
)
summary_path = reports_dir / "offline_augmentation_summary.csv"
summary_df.to_csv(summary_path, index=False)

display(summary_df)
print(f"Saved reports to: {reports_dir}")

,augmentation_type,status,count
0,blur_compression,generated,38
1,ir,generated,95
2,sunlight,generated,57


Saved reports to: C:\Github\smart-factory-safety-monitoring-system\ppe-detection\v2_pipeline\reports\augmentation


The individual CSV files are detailed row-level audit logs: each selected source image gets a generated, skipped, warning, or failed status with paths and notes.

The summary CSV condenses those row-level reports by augmentation type and status, which is useful for quickly confirming whether the configured ratios produced the expected amount of data.


## Final Counts


In [7]:
def generated_count(report: pd.DataFrame) -> int:
    return int(report["status"].eq("generated").sum())


final_counts = pd.DataFrame(
    [
        {"item": "original train images", "count": len(train_image_paths)},
        {"item": "IR images generated", "count": generated_count(ir_report)},
        {
            "item": "sunlight images generated",
            "count": generated_count(sunlight_report),
        },
        {
            "item": "blur/compression images generated",
            "count": generated_count(blur_compression_report),
        },
        {
            "item": "skipped images",
            "count": int(all_reports["status"].eq("skipped").sum()),
        },
    ]
)
display(final_counts)

,item,count
0,original train images,380
1,IR images generated,95
2,sunlight images generated,57
3,blur/compression images generated,38
4,skipped images,0


These final counts are the quick notebook-level checkpoint. The generated counts should usually match the intended counts unless files already existed, labels were missing, or an image failed to process.

Skipped rows are not always errors. They are expected when rerunning the notebook with `overwrite_existing_augmented = False` after outputs have already been generated.


## Visual Verification Examples

Save small side-by-side examples for report writing and quick sanity checks.


In [8]:
figures_dir = V2_ROOT / "reports" / "figures"
figures_dir.mkdir(parents=True, exist_ok=True)


def save_examples(
    report: pd.DataFrame, output_path: Path, title: str, max_examples: int = 3
) -> None:
    generated = report.loc[report["status"].eq("generated")].head(max_examples)
    if generated.empty:
        print(f"No generated samples available for {title}; skipping figure.")
        return

    fig, axes = plt.subplots(len(generated), 2, figsize=(8, 3 * len(generated)))
    if len(generated) == 1:
        axes = [axes]

    for row_axes, (_, row) in zip(axes, generated.iterrows()):
        original = Image.open(row["original_image_path"]).convert("RGB")
        augmented = Image.open(row["augmented_image_path"]).convert("RGB")
        row_axes[0].imshow(original)
        row_axes[0].set_title("Original train")
        row_axes[0].axis("off")
        row_axes[1].imshow(augmented)
        row_axes[1].set_title(title)
        row_axes[1].axis("off")

    fig.tight_layout()
    fig.savefig(output_path, dpi=140)
    plt.close(fig)
    print(f"Saved {output_path}")


save_examples(
    ir_report, figures_dir / "ir_augmentation_examples.png", "IR augmentation"
)
save_examples(
    sunlight_report,
    figures_dir / "sunlight_augmentation_examples.png",
    "Sunlight augmentation",
)
save_examples(
    blur_compression_report,
    figures_dir / "blur_compression_examples.png",
    "Blur/compression augmentation",
)

Saved C:\Github\smart-factory-safety-monitoring-system\ppe-detection\v2_pipeline\reports\figures\ir_augmentation_examples.png
Saved C:\Github\smart-factory-safety-monitoring-system\ppe-detection\v2_pipeline\reports\figures\sunlight_augmentation_examples.png
Saved C:\Github\smart-factory-safety-monitoring-system\ppe-detection\v2_pipeline\reports\figures\blur_compression_examples.png


This optional visual check saves side-by-side examples for each augmentation type. The goal is not to inspect every sample, but to catch obviously unrealistic settings such as images that are too dark, too white, or excessively blurred.

The figures are generated reports and remain ignored by Git, so they can be recreated whenever the augmentation parameters change.


## Checklist Before Notebook 05

- Augmentations were generated only from `splits_original/train`.
- Labels were copied unchanged and share matching augmented base names.
- Validation and test splits were not modified.
- CSV reports were saved under `reports/augmentation`.
- Example figures were reviewed if generated.
- Continue to Notebook 05 to build ablation experiment datasets.
